In [ ]:
%pip install gensim
%pip install pyldavis

In [ ]:
import os
from toolbox_tm import *
from pathlib import Path

PATH_100_NOVELS = Path ("../corpora/100_English_Novels/raw/corpus")
PATH_100_NOVELS_LEMMA = Path ("../corpora/100_English_Novels/lemmatized/corpus")
PATH_SMALL_COLLECTION = Path ('../corpora/A_Small_Collection_of_British_Fiction/lemmatized/corpus')

#### Loading the corpus

In [ ]:
processed = load_corpus (PATH_SMALL_COLLECTION, exclude_stop_words= True, drop_ners = True)

# Quick look at what one document looks like after processing
first_doc = next(iter(processed))
print(f"\nFirst document: {first_doc}")
print(f"First 20 tokens: {', '.join (processed[first_doc][:20])}")

#### Building the vocabulary

In [ ]:
dictionary = build_dictionary (processed, min_df = 2, max_df = 0.8)

#### Fitting the LDA model

In [ ]:
lda_model = fit_lda (processed, dictionary, num_topics = 30, passes=15)

#### Visualizing results

In [ ]:
top_words_per_topic(lda_model, n = 15)

In [ ]:
topic_wordclouds (lda_model, max_words = 30, colormap = 'plasma')

In [ ]:
top_documents_per_topic (lda_model, processed, dictionary, n=3)

In [ ]:
fig = document_topic_heatmap(lda_model, processed, dictionary)

In [ ]:
target_range = [10, 20, 30]
find_optimal_coherence (processed, dictionary, target_range)

In [ ]:
def generate_html (model, processed, dictionary, filename = 'lda_visualization.html'):
    documents = list(processed.values())
    bow_corpus = [dictionary.doc2bow(doc) for doc in documents]
    vis_data = gensimvis.prepare (
        topic_model = model,      # Your trained Gensim LdaModel or LdaMulticore
        corpus = bow_corpus,              # The Bag-of-Words corpus used for training
        dictionary = dictionary,      # The Gensim Dictionary object
        sort_topics = True            # Sorts topics by total document frequency
    )

    # 2. Display the interactive chart directly inside a Jupyter Notebook
    pyLDAvis.enable_notebook ()
    pyLDAvis.display (vis_data)

    # 3. Alternative: Save the visualization as a standalone HTML file
    # This is perfect for sharing results with colleagues or embedding in a research folder
    pyLDAvis.save_html (vis_data, filename)
generate_html (lda_model, processed, dictionary)